# HyperKvasir generative VQA (BLIP-2 zero-shot)

Goal: run a zero-shot generative VQA model (BLIP-2) on HyperKvasir images with the prompt “What finding is shown?” and compare predicted text to ground-truth labels. Outputs: predictions CSV + simple accuracy match stats in `/out/blip2_zero_shot/`.

How to read results:
- Overall accuracy: percent of samples where the normalized model answer matches the ground-truth label name.
- Per-class hits: see where the model guesses the right finding; classes with 0 hits are clear misses.
- Predictions CSV: inspect raw generations if you want to refine the prompt.

In [1]:
import os
from pathlib import Path
import json
import random
import re

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, AutoTokenizer, AutoImageProcessor

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
# Paths & config
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"
OUT_DIR = DATA_ROOT / "4_generative_vqa" / "out" / "blip2_zero_shot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip2-opt-2.7b"  
SAMPLE_N = None  # set None to evaluate all rows in the test split
MAX_NEW_TOKENS = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Output dir:", OUT_DIR)


Device: cuda
Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/HyperKvasir
Output dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/HyperKvasir/4_generative_vqa/out/blip2_zero_shot


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p)

if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
# use test split
meta = meta[meta["split"] == "test"].reset_index(drop=True)
if SAMPLE_N is not None:
    meta = meta.sample(min(SAMPLE_N, len(meta)), random_state=SEED).reset_index(drop=True)

print("Eval rows:", len(meta))
print(meta.head()[["img_id", "label_name", "image_path"]])


Eval rows: 1065
        img_id              label_name  \
0  test_000000                barretts   
1  test_000001                barretts   
2  test_000002                barretts   
3  test_000003                barretts   
4  test_000004  barretts-short-segment   

                                          image_path  
0  /home/aristotle/Desktop/rag-vqa-medical/Protot...  
1  /home/aristotle/Desktop/rag-vqa-medical/Protot...  
2  /home/aristotle/Desktop/rag-vqa-medical/Protot...  
3  /home/aristotle/Desktop/rag-vqa-medical/Protot...  
4  /home/aristotle/Desktop/rag-vqa-medical/Protot...  


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
processor = Blip2Processor(image_processor=image_processor, tokenizer=tokenizer)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model = model.to(DEVICE)
model.eval()
print("Loaded BLIP-2 (slow tokenizer)")


/home/aristotle/anaconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Loaded BLIP-2 (slow tokenizer)


In [5]:
def normalize_answer(text: str) -> str:
    t = str(text).lower().strip()
    # keep alphanumerics and hyphens/space
    t = re.sub(r"[^a-z0-9\-\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def predict_row(row):
    img = Image.open(row["image_path"]).convert("RGB")
    prompt = "Question: What finding is shown? Answer:"
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    ans = processor.decode(out[0], skip_special_tokens=True)
    return ans

preds = []
for _, r in tqdm(meta.iterrows(), total=len(meta), desc="BLIP-2 eval"):
    ans = predict_row(r)
    preds.append(ans)

meta["pred_raw"] = preds
meta["pred_norm"] = meta["pred_raw"].apply(normalize_answer)
meta["label_norm"] = meta["label_name"].apply(normalize_answer)
meta["correct"] = meta["pred_norm"] == meta["label_norm"]

acc = meta["correct"].mean()
print("Accuracy (exact label match):", acc)


BLIP-2 eval:   0%|          | 0/1065 [00:00<?, ?it/s]

Accuracy (exact label match): 0.0


In [6]:
# Per-class hits
per_class = meta.groupby("label_name")["correct"].agg(["mean", "count", "sum"]).rename(columns={"mean": "acc", "count": "support", "sum": "hits"})
print("Per-class (acc, hits, support) head:")
display(per_class.sort_values("acc", ascending=False).head())
display(per_class.sort_values("acc", ascending=True).head())


Per-class (acc, hits, support) head:


,acc,support,hits
label_name,,,
barretts,0.0,4,0
polyps,0.0,103,0
ulcerative-colitis-grade-3,0.0,13,0
ulcerative-colitis-grade-2-3,0.0,3,0
ulcerative-colitis-grade-2,0.0,44,0


,acc,support,hits
label_name,,,
barretts,0.0,4,0
ulcerative-colitis-grade-2-3,0.0,3,0
ulcerative-colitis-grade-2,0.0,44,0
ulcerative-colitis-grade-1-2,0.0,1,0
ulcerative-colitis-grade-1,0.0,20,0


## Label mapping (synonyms & fuzzy matching)
BLIP-2 generations often use variants (e.g., "polyp", "dyed polyp", "z line"). We map normalized predictions to the closest label using:
- Exact/alias lookup
- Substring match
- Fuzzy match (difflib)
Then recompute accuracy and per-class hits.


In [7]:
# Label mapping (synonyms & fuzzy matching)
# Ensure earlier cells have run
if 'id_to_name' not in globals() or 'meta' not in globals():
    raise RuntimeError("Run the data loading + BLIP-2 generation cells before this mapping step (meta/id_to_name missing).")

import difflib

# Build alias list for each label
alias_map = {
    "barretts": ["barrett", "barretts esophagus", "barrett's"],
    "barretts-short-segment": ["barretts short", "short segment barretts"],
    "bbps-0-1": ["bbps 0-1", "bbps 0", "bbps1"],
    "bbps-2-3": ["bbps 2-3", "bbps2", "bbps3"],
    "cecum": ["cecum", "cecim"],
    "dyed-lifted-polyps": ["dyed polyp", "dyed-lifted polyp", "lifted polyp"],
    "dyed-resection-margins": ["resection margin", "dyed resection", "resection"],
    "esophagitis-a": ["esophagitis", "esophagitis a"],
    "esophagitis-b-d": ["esophagitis b", "esophagitis d"],
    "hemorrhoids": ["hemorrhoid", "piles"],
    "ileum": ["ileum", "ileal"],
    "impacted-stool": ["stool", "impacted stool"],
    "polyps": ["polyp", "polyps"],
    "pylorus": ["pylorus", "pyloric"],
    "retroflex-rectum": ["retroflex rectum", "rectum retroflex"],
    "retroflex-stomach": ["retroflex stomach", "stomach retroflex"],
    "ulcerative-colitis-grade-0-1": ["uc grade 0", "uc grade 1"],
    "ulcerative-colitis-grade-1": ["uc grade 1"],
    "ulcerative-colitis-grade-1-2": ["uc grade 1", "uc grade 2"],
    "ulcerative-colitis-grade-2": ["uc grade 2"],
    "ulcerative-colitis-grade-2-3": ["uc grade 2", "uc grade 3"],
    "ulcerative-colitis-grade-3": ["uc grade 3"],
    "z-line": ["z line", "z-line", "zline"],
}

labels = list(id_to_name.values())
labels_norm = [ln.lower() for ln in labels]
alias_lookup = {}
for lbl, aliases in alias_map.items():
    for a in aliases:
        alias_lookup[a.lower()] = lbl


def map_pred_to_label(pred_norm: str):
    pn = str(pred_norm).lower().strip()
    if pn in labels_norm:
        return labels[labels_norm.index(pn)]
    if pn in alias_lookup:
        return alias_lookup[pn]
    for lbl in labels:
        if lbl in pn or pn in lbl:
            return lbl
    match = difflib.get_close_matches(pn, labels_norm, n=1, cutoff=0.6)
    if match:
        return labels[labels_norm.index(match[0])]
    return None

meta["pred_mapped"] = meta["pred_norm"].apply(map_pred_to_label)
meta["correct_mapped"] = meta["pred_mapped"] == meta["label_name"]

acc_mapped = meta["correct_mapped"].mean()
print("Mapped accuracy (alias/fuzzy):", acc_mapped)

per_class_mapped = meta.groupby("label_name")["correct_mapped"].agg(["mean", "count", "sum"]).rename(columns={"mean": "acc", "count": "support", "sum": "hits"})
print("Per-class (mapped) head:")
display(per_class_mapped.sort_values("acc", ascending=False).head())
display(per_class_mapped.sort_values("acc", ascending=True).head())

# Save mapped outputs
meta.to_csv(OUT_DIR / "predictions_mapped.csv", index=False)
per_class_mapped.to_csv(OUT_DIR / "per_class_mapped.csv")


Mapped accuracy (alias/fuzzy): 0.0
Per-class (mapped) head:


,acc,support,hits
label_name,,,
barretts,0.0,4,0
polyps,0.0,103,0
ulcerative-colitis-grade-3,0.0,13,0
ulcerative-colitis-grade-2-3,0.0,3,0
ulcerative-colitis-grade-2,0.0,44,0


,acc,support,hits
label_name,,,
barretts,0.0,4,0
ulcerative-colitis-grade-2-3,0.0,3,0
ulcerative-colitis-grade-2,0.0,44,0
ulcerative-colitis-grade-1-2,0.0,1,0
ulcerative-colitis-grade-1,0.0,20,0


In [8]:
# Save predictions
meta.to_csv(OUT_DIR / "predictions.csv", index=False)
per_class.to_csv(OUT_DIR / "per_class.csv")

print("Saved outputs to", OUT_DIR)


Saved outputs to /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/HyperKvasir/4_generative_vqa/out/blip2_zero_shot


## CLIP zero-shot label matching (closed label set)
Instead of free-form generation, use CLIP embeddings to pick the closest of the 23 label names. This provides a zero-shot classifier baseline that should yield non-zero accuracy.


In [9]:
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F

clip_name = "openai/clip-vit-base-patch32"
clip_proc = CLIPProcessor.from_pretrained(clip_name)
clip = CLIPModel.from_pretrained(clip_name).to(DEVICE)
clip.eval()

# Prepare label texts and embeddings
label_texts = list(id_to_name.values())
with torch.no_grad():
    text_tokens = clip_proc(text=label_texts, return_tensors="pt", padding=True).to(DEVICE)
    text_feats = clip.get_text_features(**text_tokens)
    text_feats = F.normalize(text_feats, dim=-1)

# Image->label matching
img_feats_list = []
with torch.no_grad():
    for _, row in tqdm(meta.iterrows(), total=len(meta), desc="CLIP img feats"):
        img = Image.open(row["image_path"]).convert("RGB")
        inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
        img_feat = clip.get_image_features(**inputs)
        img_feat = F.normalize(img_feat, dim=-1)
        img_feats_list.append(img_feat.cpu())

img_feats = torch.cat(img_feats_list, dim=0)
sims = img_feats @ text_feats.cpu().T
pred_idx = sims.argmax(dim=1).numpy()
pred_labels = [label_texts[i] for i in pred_idx]

meta["clip_pred"] = pred_labels
meta["clip_correct"] = meta["clip_pred"] == meta["label_name"]
clip_acc = meta["clip_correct"].mean()
print("CLIP zero-shot accuracy:", clip_acc)

per_class_clip = meta.groupby("label_name")["clip_correct"].agg(["mean", "count", "sum"]).rename(columns={"mean": "acc", "count": "support", "sum": "hits"})
print("Per-class (CLIP) head:")
display(per_class_clip.sort_values("acc", ascending=False).head())
display(per_class_clip.sort_values("acc", ascending=True).head())

# Save CLIP outputs
meta.to_csv(OUT_DIR / "predictions_clip.csv", index=False)
per_class_clip.to_csv(OUT_DIR / "per_class_clip.csv")


/home/aristotle/anaconda3/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


CLIP img feats:   0%|          | 0/1065 [00:00<?, ?it/s]

CLIP zero-shot accuracy: 0.06009389671361502
Per-class (CLIP) head:


,acc,support,hits
label_name,,,
esophagitis-a,0.875000,40,35
dyed-lifted-polyps,0.240000,100,24
esophagitis-b-d,0.038462,26,1
ulcerative-colitis-grade-2,0.022727,44,1
polyps,0.019417,103,2


,acc,support,hits
label_name,,,
barretts,0.0,4,0
ulcerative-colitis-grade-2-3,0.0,3,0
ulcerative-colitis-grade-1-2,0.0,1,0
ulcerative-colitis-grade-1,0.0,20,0
ulcerative-colitis-grade-0-1,0.0,3,0
